Imports and General Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import warnings
warnings.filterwarnings('ignore')

import pathlib
import pickle
import copy
import torch
from os import listdir
from os.path import isfile, join
from scipy.stats import sem

# Interactive plots
import PyQt5
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'qt')

# Project modules (pulls in numpy, pandas, mne, sklearn, pyriemann, etc.)
from src.preprocessing import *
from src.training import *
from src.evaluation import *
from src.group_analysis import *
from src.group_analysis_full_epoch import *


# Define paths
current_path = pathlib.Path().absolute().parent
recording_path = current_path / 'Recordings'
figure_outputs_path = current_path / 'Figures'

# List available recordings
recording_files = [f for f in listdir(recording_path) if isfile(join(recording_path, f)) and '.xdf' in f]
subject_names = [r.split('_')[0] for r in recording_files]
print('Recording files:', recording_files)
print('Subject IDs:', subject_names)

if not figure_outputs_path.exists():
    print('Output folder does not exist:', figure_outputs_path)

# Electrode groups
Electorde_Groups = {
    'FP': ['Fp1', 'Fp2'],
    'AF': ['AF7', 'AF3', 'AFz', 'AF4', 'AF8'],
    'F':  ['F7', 'F5', 'F3', 'F1', 'Fz', 'F2', 'F4', 'F6', 'F8'],
    'FC': ['FC5', 'FC3', 'FC1', 'FC2', 'FC4', 'FC6'],
    'C':  ['C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6'],
    'CP': ['CP5', 'CP3', 'CP1', 'CPz', 'CP2', 'CP4', 'CP6'],
    'P':  ['P7', 'P5', 'P3', 'P1', 'Pz', 'P2', 'P4', 'P6', 'P8'],
    'PO': ['PO7', 'PO3', 'POz', 'PO4', 'PO8'],
    'O':  ['Oz', 'O2', 'O1', 'Iz']
}

# Group tracking
group_results = []

c:\Users\CensorLab\anaconda3\envs\BCIEnvironment\lib\site-packages\moabb\pipelines\__init__.py:26: ModuleNotFoundError: Tensorflow is not installed. You won't be able to use these MOABB pipelines if you attempt to do so.
  warn(


Recording files: ['AN_MI1.xdf', 'AN_MI2.xdf', 'AN_MI3.xdf', 'AN_MI4.xdf', 'DD_1.xdf', 'DD_2.xdf', 'DD_3.xdf', 'DD_Idle.xdf', 'EA_MI1.xdf', 'EA_MI2.xdf', 'EA_MI3.xdf', 'EA_MI4.xdf', 'Fudge_MI3_1.xdf', 'Fudge_MI3_2.xdf', 'Fudge_MI3_3.xdf', 'Fudge_MI3_4.xdf', 'Gilad_3rdArm_0804_1.xdf', 'Gilad_3rdArm_0804_2.xdf', 'Gilad_3rdArm_0804_4.xdf', 'Gilad_3rdArm_0804_5.xdf', 'Gilad_RSL1.xdf', 'Gilad_RSL2.xdf', 'ID_FB1.xdf', 'ID_MI1.xdf', 'ID_MI2.xdf', 'ID_MI3.xdf', 'JE_Idle.xdf', 'JE_MI1.xdf', 'JE_MI2.xdf', 'JE_MI3.xdf', 'JE_MI4.xdf', 'LD_MI1.xdf', 'LD_MI2.xdf', 'NC_MI1.xdf', 'NC_MI2.xdf', 'NC_MI3.xdf', 'NC_MI4.xdf', 'NoamV_Idle.xdf', 'NoamV_MI1.xdf', 'NoamV_MI2.xdf', 'NoamV_MI3.xdf', 'NoamV_MI4.xdf', 'Noam_3rdArm_0104_1.xdf', 'Noam_3rdArm_0104_2.xdf', 'Noam_3rdArm_0104_3.xdf', 'Noam_3rdArm_0104_4.xdf', 'NS_MI1.xdf', 'NS_MI2.xdf', 'NS_MI4.xdf', 'NS_MI5.xdf', 'NxSx_MI3x.xdf', 'NZ_MI1.xdf', 'NZ_MI2.xdf', 'NZ_MI3.xdf', 'NZ_MI4.xdf', 'OT_MI1.xdf', 'OT_MI2.xdf', 'Ron_MI_3_1.xdf', 'Ron_MI_3_2.xdf', 'Ron_

Define Subject Name

In [60]:
subject_name = 'LD' # Manually define subject name

Choose a parameter dictionary 

In [93]:
params_dict={}
params_dict['PerformCsd']=False
params_dict['PerformAvgRef']=True
Electorde_Group_Names='FC+C+CP+P'
params_dict['Electorde_Group']=[] 
for cur_elec_group_name in Electorde_Group_Names.split('+'):
    params_dict['Electorde_Group']=params_dict['Electorde_Group']+Electorde_Groups[cur_elec_group_name]
params_dict['bad_electrodes'] = get_subject_bad_electrodes (subject_name) # Manually define subject name
params_dict['filter_method']='iir'
params_dict['epoch_tmins_and_maxes_grid'] = [-5,6]
params_dict['epoch_tmin'] = -5
params_dict['epoch_tmax'] = 6
params_dict['n_components']= 8
params_dict['LowPass']=8
params_dict['HighPass']=32
params_dict['filters_bands']=[[7, 12], [12, 20], [20, 28], [28, 35]]
params_dict['augmentation_params']={'win_len': 0, 'win_step': 0.25}
params_dict['classifier_window_s']=0.2
params_dict['classifier_window_e']=4
params_dict['windowed_prediction_params']={'win_len': 2, 'win_step': 0.25}
params_dict['pipeline_name']='ts+FGDA'
params_dict['n_components_fbcsp']=8
params_dict['desired_events'] = ['MiddleHand','LeftHand','RightHand','FixatedRest','Rest']


Load XDF files and convert to combined mne Raw instance

In [94]:
xdf_files = [f for f in recording_path.glob('*.xdf') if subject_name in  f.name] # fill in name to select subject files
#OriginalRaw = Load_and_concatenate_xdf(xdf_files)
xdf_files[:]

[WindowsPath('c:/Users/CensorLab/3rd_arm_MI/Recordings/LD_MI1.xdf'),
 WindowsPath('c:/Users/CensorLab/3rd_arm_MI/Recordings/LD_MI2.xdf'),
 WindowsPath('c:/Users/CensorLab/3rd_arm_MI/Recordings/LD_MI3.xdf')]

In [102]:
epochs_list = []
filter_bank_epochs_list = []
movement_events = ['ClosePalm','OpenPalm'] 
for xdf_file in xdf_files[:-1]:
    raw=read_raw_xdf(xdf_file)
    # # Step 1: move all affected channels to temporary names
    # raw.rename_channels({
    #     'F8':   '_tmp_F8',   'F4':   '_tmp_F4',
    #     'FC2':  '_tmp_FC2',  'FT10': '_tmp_FT10',
    #     'Cz':   '_tmp_Cz',   'T8':   '_tmp_T8',
    #     'CP2':  '_tmp_CP2',  'CP6':  '_tmp_CP6',
    #     'P4':   '_tmp_P4',   'TP10': '_tmp_TP10',
    #     'P7':   '_tmp_P7',   'P3':   '_tmp_P3',
    #     'Pz':   '_tmp_Pz',   'CP1':  '_tmp_CP1',
    #     'CP5':  '_tmp_CP5',  'TP9':  '_tmp_TP9',
    # })

    # # Step 2: rename temps to their final destinations
    # raw.rename_channels({
    #     # swaps
    #     '_tmp_F8':   'F4',   '_tmp_F4':   'F8',
    #     '_tmp_FC2':  'FT10', '_tmp_FT10': 'FC2',
    #     '_tmp_Cz':   'T8',   '_tmp_T8':   'Cz',
    #     '_tmp_CP2':  'CP6',  '_tmp_CP6':  'CP2',
    #     '_tmp_P4':   'TP10', '_tmp_TP10': 'P4',
    #     # cycle: P7ג†’P3ג†’Pzג†’CP1ג†’CP5ג†’TP9ג†’P7
    #     '_tmp_P7':  'P3',
    #     '_tmp_P3':  'Pz',
    #     '_tmp_Pz':  'CP1',
    #     '_tmp_CP1': 'CP5',
    #     '_tmp_CP5': 'TP9',
    #     '_tmp_TP9': 'P7',
    # })
    #threshold_raw = filter_events_by_rating(raw,movement_events,rating_threshold = 4)
    #raw_for_ICA = raw_EEG_Preprocessing(current_path,raw,params_dict)
    filtered_raw,epoch,filter_bank_epochs,mean_across_epochs, events_trigger_dict = EEG_Preprocessing(current_path,raw,params_dict, pick_channels=True)
    epoch = remap_epoch_events_to_standard(epoch, standard_event_id, params_dict['desired_events'])
    # Update events_trigger_dict to match new labels
    params_dict['events_trigger_dict'] = {event: standard_event_id[event] for event in params_dict['desired_events']}
    events_trigger_dict = params_dict['events_trigger_dict']
    epochs_list.append(epoch)
    filter_bank_epochs_list.append(filter_bank_epochs)

print("Concatenating all preprocessed epochs...")


epochs = mne.concatenate_epochs(epochs_list, on_mismatch='warn')
filter_bank_epochs = None
epochs

Creating RawArray with float64 data, n_channels=67, n_times=396503
    Range : 0 ... 396502 =      0.000 ...   793.004 secs
Ready.

###########################################################
removing subject specific bad electrodes from the raw data

###########################################################
removing bad channels from epochs:
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.

###########################################################
filtering the data
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 1e+02 Hz

IIR filter parameters
---------------------
Butterworth bandpass non-linear phase (one-pass forward) causal filter:
- Filter order 8 (forward)
- Cutoffs at 1.00, 100.00 Hz: -3.01, -3.01 dB

Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 49 - 51 Hz

IIR filter parameters
---------------------
Butterworth bandstop non-linear phase (one-pass fo

Number of events,219
Events,FixatedRest: 20LeftHand: 30MiddleHand: 30Rest: 109RightHand: 30
Time range,-5.000 – 6.000 s
Baseline,off


Balance the classes if required

In [103]:
balanced_epochs = balance_epochs_by_subsampling(epochs, class_to_subsample='Rest')
epochs = balanced_epochs

Pre-processing

In [104]:
from sklearn.model_selection import RepeatedStratifiedKFold

In [105]:
events = epochs.events
event_dict = epochs.event_id
params_dict['events_trigger_dict'] = {key: event_dict[key] for key in event_dict.keys() if key in params_dict['desired_events']}
train_inds,validation_inds,return_dict = Split_training_validation (epochs,filter_bank_epochs, params_dict['events_trigger_dict'])
epochs_copy = epochs.copy()
returned_dict=crop_the_data(epochs_copy,train_inds,validation_inds,params_dict['classifier_window_s'],params_dict['classifier_window_e'],use_all_for_training=True) 
train_set_data_uncropped=returned_dict['train_set_data_uncropped']
epochs_cropped=returned_dict['epochs_cropped']
train_set_data=returned_dict['train_set_data']
train_set_labels=returned_dict['train_set_labels']

validation_set_labels=returned_dict['validation_set_labels']
validation_set_data_uncropped=returned_dict['validation_set_data_uncropped']
#define cv on the data: 
cv = StratifiedShuffleSplit(10, test_size=0.2, random_state=22)
cv_split = cv.split(epochs_cropped.get_data(),events[:,2])

#filter bank related:
if params_dict['pipeline_name']=='fbcsp+lda': 
    train_set_data_fb=[]
    train_set_data_uncropped_fb=[]
    validation_set_data_fb=[]
    validation_set_data_uncropped_fb=[]
    for filtered_data_band_epoch in preprocessing_dict['filter_bank_epochs']:
        returned_dict_temp=crop_the_data(filtered_data_band_epoch,train_inds,validation_inds, params_dict['classifier_window_s'],params_dict['classifier_window_e'])
        #extract the train set data: 
        train_set_data_uncropped_temp=returned_dict_temp['train_set_data_uncropped']
        train_set_data_temp=returned_dict_temp['train_set_data']
        train_set_data_fb.append(train_set_data_temp)
        train_set_data_uncropped_fb.append(train_set_data_uncropped_temp)
        #extract the validation set data: 
        validation_set_data_uncroped_temp=returned_dict_temp['validation_set_data_uncroped']
        validation_set_data_temp=returned_dict_temp['validation_set_data']
        validation_set_data_fb.append(validation_set_data_temp)
        validation_set_data_uncropped_fb.append(validation_set_data_uncroped_temp)
    #create a 4d matrix of train data:     
    train_set_data_4d_array= np.transpose(np.array(train_set_data_fb),(1,2,3,0))
    train_set_data_uncropped_4d_array=np.transpose(np.array(train_set_data_uncropped_fb),(1,2,3,0)) 
    train_set_data=train_set_data_4d_array
    train_set_data_uncropped=train_set_data_uncropped_4d_array
    #create a 4d matrix of validation data: 
    validation_set_data_4d_array= np.transpose(np.array(validation_set_data_fb),(1,2,3,0))
    validation_set_data_uncropped_4d_array=np.transpose(np.array(validation_set_data_uncropped_fb),(1,2,3,0)) 
    validation_set_data_uncropped=validation_set_data_uncropped_4d_array


augmented_x,augmented_y = augment_data(params_dict['augmentation_params'], train_set_data,train_set_labels,epochs.info['sfreq'])

triggers_label_dict={val:key for key,val in params_dict['events_trigger_dict'].items()} 
test_data_y_labels=np.array([triggers_label_dict[cur_y] for cur_y in validation_set_labels])  
augmented_y_labels=np.array([triggers_label_dict[cur_y] for cur_y in augmented_y])  

putting aside 20% of the data: trial numbers are:
 [ 84  96  43  90  47  61  79  27  19 106   5   0  23  80 119  17 112 126
  36  78  41  58  67  31  59 120]

remaining 80% of the trials go into training for cv:
 [ 18 108 101  98  15  76  30   3 123  11  68  64   7  55   9   6  39   1
  16  26  77  37 125 115   4  89  14  87  65 102  10  95  81 114 109  46
  75  83  99  88  97  29  33  53  85  20  40  12  32 107 117  21 122  71
  51  48  94  50  92  38  49  62 127  44  70 105   2 121  72 100  74   8
  34  56  35  57  63  54  13  86 118 111  22  60 129  73 103 110 116  25
  93 124  52  91  45  28  42 128  82 104  24 113  66  69]



Cross-Validation Evaluation 

In [106]:
scores_windows, folds_confusion_matrices_per_window, w_times =run_windowed_classification_aug_cv(epochs, returned_dict['epochs_cropped'], cv_split,params_dict)


In [107]:
plot_accuracy_over_time(scores_windows, w_times, params_dict, axes_handle=None)
plot_average_confusion_fixed_cv(folds_confusion_matrices_per_window, w_times, t_start=2, t_end=5)


(array([[0.44615385, 0.16538462, 0.275     , 0.02115385, 0.09230769],
        [0.08846154, 0.51666667, 0.21923077, 0.05384615, 0.12179487],
        [0.05384615, 0.12435897, 0.66410256, 0.05      , 0.10769231],
        [0.01923077, 0.08653846, 0.32884615, 0.35384615, 0.21153846],
        [0.07820513, 0.15512821, 0.16538462, 0.07564103, 0.52564103]]),
 array(['FixatedRest', 'LeftHand', 'MiddleHand', 'Rest', 'RightHand'],
       dtype='<U11'))

In [108]:
plot_average_confusion_fixed_cv(folds_confusion_matrices_per_window, w_times, t_start=-2, t_end=0)
plot_average_confusion_fixed_cv(folds_confusion_matrices_per_window, w_times, t_start=0, t_end=5)


(array([[0.35      , 0.13333333, 0.36547619, 0.04880952, 0.10238095],
        [0.09285714, 0.40079365, 0.2484127 , 0.09047619, 0.16746032],
        [0.10555556, 0.12063492, 0.56984127, 0.07460317, 0.12936508],
        [0.03571429, 0.12142857, 0.35595238, 0.25357143, 0.23333333],
        [0.10555556, 0.13809524, 0.22539683, 0.13412698, 0.3968254 ]]),
 array(['FixatedRest', 'LeftHand', 'MiddleHand', 'Rest', 'RightHand'],
       dtype='<U11'))

In [109]:
epochs

Number of events,130
Events,FixatedRest: 20LeftHand: 30MiddleHand: 30Rest: 20RightHand: 30
Time range,-5.000 – 6.000 s
Baseline,off


In [110]:
scores_windows_s1 = scores_windows
folds_cm_s1 = folds_confusion_matrices_per_window
w_times_s1 = w_times

Training

In [111]:
clf,csp,lda = classifier_training(train_set_data,train_set_labels,params_dict, BinaryClassification = False)
trained_clf = clf

In [72]:
trained_clf

Pipeline(steps=[('cov', Covariances(estimator='oas')), ('fgda', FGDA()),
                ('ts', TangentSpace()), ('scaler', StandardScaler()),
                ('lda', LinearDiscriminantAnalysis())])

In [151]:
csp.plot_patterns(epochs.info, units='Patterns (AU)', size=1.5)

AttributeError: 'NoneType' object has no attribute 'plot_patterns'

Permutation Tests

In [64]:
# Permutation test ג€” validates that the classifier learns real signal, not noise.
# score_method='majority_vote' matches run_full_epoch_classification_cv (majority vote per trial).
# With n_permutations=100 this takes ~100ֳ— the time of a single CV run.
# Use n_permutations=1000 for publication-quality p-values.

true_score, perm_scores, p_value, true_fold_cms, perm_cm_sum, perm_classes = run_permutation_test(
    epochs,
    returned_dict['epochs_cropped'],
    params_dict,
    n_permutations=10,
    score_method='majority_vote',   # matches run_full_epoch_classification_cv
    eval_tmin=0.0,
    eval_tmax=5.0,
)

print(f"True score:  {true_score:.3f}")
print(f"Null mean ֲ± std: {np.mean(perm_scores):.3f} ֲ± {np.std(perm_scores):.3f}")
print(f"p-value: {p_value:.4f}")

Starting permutation test: 10 permutations ֳ— 10 CV folds (score_method='majority_vote', eval window [0.0, 5.0] s). Expected runtime ג‰ˆ 10ֳ— a single CV run.


Permutations: 100%|ג–ˆג–ˆג–ˆג–ˆג–ˆג–ˆג–ˆג–ˆג–ˆג–ˆ| 10/10 [06:34<00:00, 39.47s/perm]

True score:  0.791
Null mean ֲ± std: 0.237 ֲ± 0.038
p-value: 0.0909


In [65]:
fig, ax = plot_permutation_test(
    true_score, perm_scores, p_value,
    params_dict=params_dict,
    score_tmin=0.0, score_tmax=5.0,
)

In [66]:
avg_cm, labels = plot_permutation_test_confusion(true_fold_cms, normalize=True)


In [67]:
# Side-by-side: true classifier CM vs permuted (null) CM
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

plot_permutation_test_confusion(true_fold_cms, normalize=True, axes_handle=axes[0])
axes[0].set_title(f"True classifier (acc={true_score:.2f})")

# Null distribution CM: row-normalize the accumulated sum
perm_cm_norm = perm_cm_sum / perm_cm_sum.sum(axis=1, keepdims=True)
disp = ConfusionMatrixDisplay(confusion_matrix=perm_cm_norm, display_labels=perm_classes)
disp.plot(cmap="Blues", ax=axes[1], values_format=".2f")
axes[1].grid(False)
axes[1].set_title(f"Permuted labels (null, n={len(perm_scores)})")  

plt.tight_layout()
plt.show()


In [26]:
# Full-epoch CV ג€” recreate cv_split first (generator is exhausted above)
cv_split = cv.split(epochs_cropped.get_data(), events[:, 2])
fold_accuracies, fold_confusion_matrices = run_full_epoch_classification_cv(
    epochs, returned_dict['epochs_cropped'], cv_split, params_dict, tmin=0.0, tmax=5.0
)
print(f"Full-epoch CV accuracy: {np.mean(fold_accuracies):.3f} ֲ± {np.std(fold_accuracies):.3f}")

Full-epoch CV accuracy: 0.818 ֲ± 0.045


In [66]:
# Recreate cv_split — the generator was exhausted by the cell above
cv_split = cv.split(epochs_cropped.get_data(), events[:, 2])
fold_accuracies, fold_confusion_matrices = run_full_epoch_classification_cv(
    epochs, returned_dict['epochs_cropped'], cv_split, params_dict, tmin=0.0, tmax=5.0
)
print(f"Full-epoch CV accuracy: {np.mean(fold_accuracies):.3f} ± {np.std(fold_accuracies):.3f}")
# Average confusion matrix across folds
all_cms = np.array([cm for cm, _ in fold_confusion_matrices], dtype=float)
classes = fold_confusion_matrices[0][1]
avg_cm = all_cms.mean(axis=0)
avg_cm_normalized = avg_cm / avg_cm.sum(axis=1, keepdims=True)

plot_confusion_matrix(avg_cm_normalized, classes, title="Full Epoch (0–5s) — Avg CV Confusion Matrix")

Full-epoch CV accuracy: 0.545 ± 0.052


In [39]:
from src.group_analysis_full_epoch import (
    add_subject_to_group_full_epoch,
    save_group_results_full_epoch,
    load_group_results_full_epoch,
)


In [ ]:
# Initialise once (top of the subject loop)
#group_results_full_epoch = []



In [67]:
# After each subject's run_full_epoch_classification_cv call:
add_subject_to_group_full_epoch(
    group_results_full_epoch,
    subject_name=subject_name,
    fold_accuracies=fold_accuracies,
    fold_confusion_matrices=fold_confusion_matrices,
    tmin=0.0, tmax=5.0,
    params_dict=params_dict,
    save_dir=current_path / 'Metrics/Indivuals_Full_Epoch',   # saves AN_CP_RH_LH_..._full_epoch.json
)

Added NZ: 10 folds | acc = 0.545 ± 0.052 (epoch 0.0–5.0s)
  Saved full-epoch metrics to c:\Users\CensorLab\3rd_arm_MI\Metrics\Indivuals_Full_Epoch\NZ_CP_RH_LH_FR_ts+FGDA_full_epoch.json


[{'subject_name': 'SK',
  'timestamp': '2026-05-27T08:46:00.700006',
  'tmin': 0.0,
  'tmax': 5.0,
  'fold_accuracies': [0.5454545454545454,
   0.48484848484848486,
   0.48484848484848486,
   0.42424242424242425,
   0.42424242424242425,
   0.6363636363636364,
   0.3939393939393939,
   0.48484848484848486,
   0.48484848484848486,
   0.3939393939393939],
  'fold_confusion_matrices': [(array([[3, 0, 4, 2],
           [1, 4, 1, 0],
           [2, 0, 4, 3],
           [1, 0, 1, 7]], dtype=int64),
    array(['ClosePalm', 'FixatedRest', 'LeftHand', 'RightHand'], dtype='<U11')),
   (array([[3, 0, 4, 2],
           [0, 3, 2, 1],
           [5, 0, 4, 0],
           [2, 0, 1, 6]], dtype=int64),
    array(['ClosePalm', 'FixatedRest', 'LeftHand', 'RightHand'], dtype='<U11')),
   (array([[3, 0, 5, 1],
           [0, 3, 1, 2],
           [2, 0, 7, 0],
           [3, 0, 3, 3]], dtype=int64),
    array(['ClosePalm', 'FixatedRest', 'LeftHand', 'RightHand'], dtype='<U11')),
   (array([[2, 0, 5, 2],
     

In [68]:


# Bar chart: one bar per subject, group mean band, chance line
plot_accuracy_group_full_epoch(group_results_full_epoch, n_classes=4)

# Group-averaged confusion matrix across all subjects & folds
plot_average_confusion_full_epoch_group(group_results_full_epoch, normalize=True)


(array([[0.57903226, 0.05      , 0.15967742, 0.21129032],
        [0.11428571, 0.63333333, 0.14047619, 0.11190476],
        [0.11904762, 0.05873016, 0.65873016, 0.16349206],
        [0.11111111, 0.03333333, 0.17301587, 0.68253968]]),
 array(['ClosePalm', 'FixatedRest', 'LeftHand', 'RightHand'], dtype='<U11'))

In [93]:
add_subject_to_group(group_results, subject_name, scores_windows_s1, folds_cm_s1, w_times_s1, 
                     params_dict=params_dict, save_dir=current_path / 'Metrics/Individuals' )

Added NS: 10 folds x 37 windows
  Saved metrics to c:\Users\CensorLab\3rd_arm_MI\Metrics\Individuals\NS_CP_RH_LH_ts+FGDA.json


[{'subject_name': 'ID',
  'timestamp': '2026-05-26T18:08:44.082690',
  'scores_windows': [[0.48148148148148145,
    0.5555555555555556,
    0.5555555555555556,
    0.5555555555555556,
    0.48148148148148145,
    0.4074074074074074,
    0.4444444444444444,
    0.4074074074074074,
    0.48148148148148145,
    0.4444444444444444,
    0.3333333333333333,
    0.4074074074074074,
    0.3333333333333333,
    0.3333333333333333,
    0.3333333333333333,
    0.37037037037037035,
    0.4074074074074074,
    0.4074074074074074,
    0.4074074074074074,
    0.5555555555555556,
    0.6666666666666666,
    0.7037037037037037,
    0.6666666666666666,
    0.7407407407407407,
    0.7037037037037037,
    0.6666666666666666,
    0.7037037037037037,
    0.7037037037037037,
    0.6666666666666666,
    0.5925925925925926,
    0.6666666666666666,
    0.5925925925925926,
    0.5925925925925926,
    0.7407407407407407,
    0.6296296296296297,
    0.5925925925925926,
    0.5925925925925926],
   [0.14814814814814

In [94]:

# After all subjects are processed:
print(f"Group results collected for {len(group_results)} subjects:")
for entry in group_results:
    print(f"  - {entry['subject_name']}: {len(entry['scores_windows'])} folds")

# Save group results to disk (note: confusion matrices are NOT saved, only metadata)
save_group_results(group_results, current_path / 'Metrics/Group')

# Plotting group results
# Plot 1: Group mean accuracy over time
plot_accuracy_over_time_group(group_results,n_classes=3, figsize=(12, 5))

# Plot 2: Group average confusion matrix (all windows, all subjects)
#plot_average_confusion_fixed_cv_group(group_results, window_index=None, normalize=True)

# Or focus on specific time windows (e.g., windows 20-33)
plot_average_confusion_fixed_cv_group(group_results, w_times,t_start=2, t_end=5, normalize=True)

print("ג“ Group analysis complete!")


Group results collected for 6 subjects:
  - ID: 10 folds
  - AN: 10 folds
  - NZ: 10 folds
  - EA: 10 folds
  - SK: 10 folds
  - NS: 10 folds
Saved 6 subjects to c:\Users\CensorLab\3rd_arm_MI\Metrics\Group\group_ID_AN_NZ_EA_SK_NS_CP_RH_LH_ts+FGDA.json
ג“ Group analysis complete!


In [100]:
plot_average_confusion_fixed_cv_group(group_results, w_times,t_start=0, t_end=2, normalize=True)


(array([[0.49354839, 0.33781362, 0.16863799],
        [0.3473545 , 0.53015873, 0.12248677],
        [0.41931217, 0.17460317, 0.40608466]]),
 array(['ClosePalm', 'FixatedRest', 'Rest'], dtype='<U11'))

In [84]:
import json

file_path = current_path / 'Metrics/Group/group_NS_AN_EA_NZ_SK_ID_CP_LH_RH_FR_ts+FGDA.json'

with open(file_path, 'r') as f:
    group_results = json.load(f)

print(f"Loaded data for {len(group_results)} subjects.")

Loaded data for 6 subjects.


In [83]:
group_results

[{'subject_name': 'NS',
  'timestamp': '2026-05-24T15:22:19.573695',
  'scores_windows': [[0.3235294117647059,
    0.17647058823529413,
    0.20588235294117646,
    0.29411764705882354,
    0.3235294117647059,
    0.3235294117647059,
    0.29411764705882354,
    0.2647058823529412,
    0.29411764705882354,
    0.29411764705882354,
    0.29411764705882354,
    0.23529411764705882,
    0.29411764705882354,
    0.14705882352941177,
    0.2647058823529412,
    0.23529411764705882,
    0.3235294117647059,
    0.3235294117647059,
    0.4411764705882353,
    0.6470588235294118,
    0.6764705882352942,
    0.7058823529411765,
    0.7647058823529411,
    0.6764705882352942,
    0.7058823529411765,
    0.7352941176470589,
    0.7352941176470589,
    0.7058823529411765,
    0.7352941176470589,
    0.6470588235294118,
    0.6470588235294118,
    0.6764705882352942,
    0.6176470588235294,
    0.5882352941176471,
    0.5294117647058824,
    0.47058823529411764,
    0.5294117647058824],
   [0.323529

In [77]:

# Plotting group results
# Plot 1: Group mean accuracy over time
plot_accuracy_over_time_group(group_results,n_classes = 5, figsize=(12, 5))

# Plot 2: Group average confusion matrix (all windows, all subjects)
#plot_average_confusion_fixed_cv_group(group_results, window_index=None, normalize=True)

# Or focus on specific time windows (e.g., windows 20-33)
plot_average_confusion_fixed_cv_group(group_results, window_index=range(20, 33), normalize=True)

print("ג“ Group analysis complete!")

TypeError: plot_average_confusion_fixed_cv_group() got an unexpected keyword argument 'window_index'

In [74]:
plot_average_confusion_fixed_cv_group(group_results, window_index=range(20, 33), normalize=True)


TypeError: plot_average_confusion_fixed_cv_group() got an unexpected keyword argument 'window_index'

In [ ]:
len(folds_confusion_matrices_per_window[0])

49

Sanity check on unhandled recording: 

In [293]:

xdf_files[-1:]


[WindowsPath('c:/Users/CensorLab/3rd_arm_MI/Recordings/NS_MI5.xdf')]

In [350]:
epochs_list = []
filter_bank_epochs_list = []
movement_events = ['ClosePalm','OpenPalm'] 
for xdf_file in xdf_files[-1:]:
    raw=read_raw_xdf(xdf_file)
    # Step 1: move all affected channels to temporary names
    raw.rename_channels({
        'F8':   '_tmp_F8',   'F4':   '_tmp_F4',
        'FC2':  '_tmp_FC2',  'FT10': '_tmp_FT10',
        'Cz':   '_tmp_Cz',   'T8':   '_tmp_T8',
        'CP2':  '_tmp_CP2',  'CP6':  '_tmp_CP6',
        'P4':   '_tmp_P4',   'TP10': '_tmp_TP10',
        'P7':   '_tmp_P7',   'P3':   '_tmp_P3',
        'Pz':   '_tmp_Pz',   'CP1':  '_tmp_CP1',
        'CP5':  '_tmp_CP5',  'TP9':  '_tmp_TP9',
    })

    # Step 2: rename temps to their final destinations
    raw.rename_channels({
        # swaps
        '_tmp_F8':   'F4',   '_tmp_F4':   'F8',
        '_tmp_FC2':  'FT10', '_tmp_FT10': 'FC2',
        '_tmp_Cz':   'T8',   '_tmp_T8':   'Cz',
        '_tmp_CP2':  'CP6',  '_tmp_CP6':  'CP2',
        '_tmp_P4':   'TP10', '_tmp_TP10': 'P4',
        # cycle: P7ג†’P3ג†’Pzג†’CP1ג†’CP5ג†’TP9ג†’P7
        '_tmp_P7':  'P3',
        '_tmp_P3':  'Pz',
        '_tmp_Pz':  'CP1',
        '_tmp_CP1': 'CP5',
        '_tmp_CP5': 'TP9',
        '_tmp_TP9': 'P7',
    })
    #threshold_raw = filter_events_by_rating(raw,movement_events,rating_threshold = 4)
    _,epoch,filter_bank_epochs,mean_across_epochs, events_trigger_dict = EEG_Preprocessing(current_path,raw,params_dict)
    epoch = remap_epoch_events_to_standard(epoch, standard_event_id, params_dict['desired_events'])
    # Update events_trigger_dict to match new labels
    params_dict['events_trigger_dict'] = {event: standard_event_id[event] for event in params_dict['desired_events']}
    events_trigger_dict = params_dict['events_trigger_dict']
    epochs_list.append(epoch)
    filter_bank_epochs_list.append(filter_bank_epochs)

print("Concatenating all preprocessed epochs...")


epochs = mne.concatenate_epochs(epochs_list, on_mismatch='warn')
filter_bank_epochs = None

Creating RawArray with float64 data, n_channels=67, n_times=284006
    Range : 0 ... 284005 =      0.000 ...   568.010 secs
Ready.

###########################################################
removing subject specific bad electrodes from the raw data

###########################################################
removing bad channels from epochs:
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.

###########################################################
filtering the data
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 1e+02 Hz

IIR filter parameters
---------------------
Butterworth bandpass non-linear phase (one-pass forward) causal filter:
- Filter order 8 (forward)
- Cutoffs at 1.00, 100.00 Hz: -3.01, -3.01 dB

Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 49 - 51 Hz

IIR filter parameters
---------------------
Butterworth bandstop non-linear phase (one-pass fo

In [351]:
Raw_for_analysis = mne.preprocessing.compute_current_source_density(raw) ## Compute CSD
Raw_for_analysis = Raw_for_analysis.filter(8, 35, method='fir')
original_ann = Raw_for_analysis.annotations
event_ann = expand_triggers_to_events(original_ann,Raw_for_analysis)
Raw_for_analysis.set_annotations(event_ann)

Fitted sphere radius:         95.0 mm
Origin head coordinates:      0.0 -0.0 0.0 mm
Origin device coordinates:    0.0 -0.0 0.0 mm
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 35 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 35.00 Hz
- Upper transition bandwidth: 8.75 Hz (-6 dB cutoff frequency: 39.38 Hz)
- Filter length: 825 samples (1.650 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


Measurement date,Unknown
Experimenter,Unknown
Participant,Unknown
Digitized points,68 points
Good channels,62 Current source density
Bad channels,None
EOG channels,Not available
ECG channels,Not available
Sampling frequency,500.00 Hz
Highpass,8.00 Hz
Lowpass,35.00 Hz


In [352]:
epochs

Number of events,77
Events,ClosePalm: 11FixatedRest: 7LeftHand: 14Rest: 38RightHand: 7
Time range,-5.000 – 6.000 s
Baseline,off


In [353]:
trained_clf=clf
#data_to_predict=(data_set_fb_4d_array[inds,:])
data_to_predict=epochs[:].copy().crop(tmin=1,tmax=3
                                      ).get_data()
thresholded_prediction=trained_clf.decision_function(data_to_predict)
prediction=trained_clf.predict(data_to_predict)

#note that here you can decide on which thresholds to use to better optimize your "real" usecase
#thresholded_prediction
yhat=trained_clf.predict(data_to_predict)
lr_probs = trained_clf.predict_proba(data_to_predict)

In [354]:
# %%


# Function to get the list of event names for each epoch
# Function to compare actual event names with predicted event names and summarize mismatches
# Example usage:
# Assume `epochs` is your mne.Epochs object, and `predictions` is the model output

actual_events = get_epoch_events(epochs[:])  # Actual events from the epochs
predicted_events = np.array(prediction)  # Replace with your model predictions

# Compare actual events to predicted events
result = compare_events(actual_events, predicted_events)

# Output the comparison result
print(f"Accuracy: {result['accuracy']:.2f}")
if result['mismatch_details']:
    print("Mismatched epochs:")
    for mismatch in result['mismatch_details']:
        print(mismatch)
    
    print("\nMismatch Summary:")
    for mismatch_case, count in result['mismatch_counts'].items():
        print(f"{mismatch_case}: {count} occurrence(s)")
else:
    print("No mismatches!")

Accuracy: 0.86
Mismatched epochs:
3 - LeftHand (actual), RightHand (predicted)
15 - LeftHand (actual), RightHand (predicted)
16 - Rest (actual), ClosePalm (predicted)
25 - RightHand (actual), ClosePalm (predicted)
33 - LeftHand (actual), RightHand (predicted)
39 - LeftHand (actual), RightHand (predicted)
42 - Rest (actual), ClosePalm (predicted)
47 - FixatedRest (actual), ClosePalm (predicted)
53 - FixatedRest (actual), Rest (predicted)
57 - LeftHand (actual), RightHand (predicted)
63 - LeftHand (actual), RightHand (predicted)

Mismatch Summary:
LeftHand -> RightHand: 6 occurrence(s)
Rest -> ClosePalm: 2 occurrence(s)
RightHand -> ClosePalm: 1 occurrence(s)
FixatedRest -> ClosePalm: 1 occurrence(s)
FixatedRest -> Rest: 1 occurrence(s)


In [355]:
# Function to compare events excluding Rest
# Example usage:
result_without_rest = compare_events_without_rest(actual_events, predicted_events, rest_labels=('FixatedRest','Rest'))

# Output the comparison result
print(f"\n{'='*50}")
print(f"ACCURACY WITHOUT REST")
print(f"{'='*50}")
print(f"Accuracy (excluding Rest): {result_without_rest['accuracy']:.2f}")
print(f"Total non-Rest epochs evaluated: {result_without_rest['total_non_rest_epochs']}")

if result_without_rest['mismatch_details']:
    print("\nMismatched epochs (non-Rest only):")
    for mismatch in result_without_rest['mismatch_details'][:10]:  # Show first 10
        print(mismatch)
    if len(result_without_rest['mismatch_details']) > 10:
        print(f"... and {len(result_without_rest['mismatch_details']) - 10} more")
    
    print("\nMismatch Summary (non-Rest only):")
    for mismatch_case, count in result_without_rest['mismatch_counts'].items():
        print(f"{mismatch_case}: {count} occurrence(s)")
else:
    print("No mismatches on non-Rest epochs!")


ACCURACY WITHOUT REST
Accuracy (excluding Rest): 0.78
Total non-Rest epochs evaluated: 32

Mismatched epochs (non-Rest only):
3 - LeftHand (actual), RightHand (predicted)
15 - LeftHand (actual), RightHand (predicted)
25 - RightHand (actual), ClosePalm (predicted)
33 - LeftHand (actual), RightHand (predicted)
39 - LeftHand (actual), RightHand (predicted)
57 - LeftHand (actual), RightHand (predicted)
63 - LeftHand (actual), RightHand (predicted)

Mismatch Summary (non-Rest only):
LeftHand -> RightHand: 6 occurrence(s)
RightHand -> ClosePalm: 1 occurrence(s)


In [356]:
def compare_events_without_rest(actual_events, predicted_events, rest_labels=('FixatedRest','Rest')):
    """Compare actual vs predicted events, excluding *rest_labels* epochs from accuracy.

    Parameters
    ----------
    actual_events : array-like of str
        Ground-truth event labels.
    predicted_events : array-like of str
        Predicted event labels (same length).
    rest_labels : str or iterable of str
        Label(s) to exclude from the accuracy calculation. A single string is
        also accepted for backwards compatibility.

    Returns
    -------
    dict with keys 'accuracy', 'total_non_rest_epochs', 'mismatch_details', 'mismatch_counts'.
    """
    # Allow a single string for backwards compatibility
    if isinstance(rest_labels, str):
        rest_labels = (rest_labels,)
    rest_labels = set(rest_labels)

    non_rest_mask = ~np.isin(actual_events, list(rest_labels))
    non_rest_indices = np.where(non_rest_mask)[0]

    actual_non_rest = actual_events[non_rest_mask]
    predicted_non_rest = predicted_events[non_rest_mask]

    comparison = actual_non_rest == predicted_non_rest
    accuracy = np.mean(comparison) if len(comparison) else np.nan
    mismatches = np.where(~comparison)[0]

    mismatch_details = []
    mismatch_counts = Counter()
    for mismatch_idx in mismatches:
        original_idx = non_rest_indices[mismatch_idx]
        actual = actual_non_rest[mismatch_idx]
        predicted = predicted_non_rest[mismatch_idx]
        mismatch_details.append(f"{original_idx + 1} - {actual} (actual), {predicted} (predicted)")
        mismatch_counts[f"{actual} -> {predicted}"] += 1

    return {
        'accuracy': accuracy,
        'total_non_rest_epochs': len(actual_non_rest),
        'mismatch_details': mismatch_details,
        'mismatch_counts': mismatch_counts,
    }

In [357]:
# Sanity check - windowed evaluation on trained classifier
scores_windows, confusion_matrices_per_window, w_times = sanity_check_trained_clf(
    trained_clf, epochs, params_dict
)

# Accuracy over time
plot_accuracy_over_time(scores_windows, w_times, params_dict)

# Average confusion matrix between time points

plot_average_confusion_fixed_cv(confusion_matrices_per_window, w_times, t_start=-2, t_end=0)

(array([[0.21212121, 0.13131313, 0.01010101, 0.63636364, 0.01010101],
        [0.        , 0.34920635, 0.        , 0.65079365, 0.        ],
        [0.03968254, 0.11904762, 0.        , 0.79365079, 0.04761905],
        [0.26315789, 0.15497076, 0.2251462 , 0.0994152 , 0.25730994],
        [0.20634921, 0.20634921, 0.01587302, 0.52380952, 0.04761905]]),
 array(['ClosePalm', 'FixatedRest', 'LeftHand', 'Rest', 'RightHand'],
       dtype='<U11'))

In [86]:
# Predict on cropped epochs
data_to_predict = epochs[:].copy().crop(tmin=1, tmax=3).get_data()
lr_probs = trained_clf.predict_proba(data_to_predict)
prediction = trained_clf.predict(data_to_predict)
actual_events = get_epoch_events(epochs[:])
class_names = trained_clf.classes_

# --- Probability heatmap per trial, sorted by true class ---
sort_idx = np.argsort(actual_events)
probs_sorted = lr_probs[sort_idx]
actual_sorted = actual_events[sort_idx]

fig, ax = plt.subplots(figsize=(6, max(4, len(actual_sorted) * 0.18)))
im = ax.imshow(probs_sorted, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax.set_yticks(range(len(actual_sorted)))
ax.set_yticklabels([f"{i+1} ({actual_sorted[i]})" for i in range(len(actual_sorted))], fontsize=7)
ax.set_xticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_xlabel('Predicted Class Probability')
ax.set_ylabel('Trial (sorted by true class)')
ax.set_title('Per-Trial Prediction Probabilities')
fig.colorbar(im, ax=ax, label='Probability')
plt.tight_layout()


In [87]:
# --- Mean probability matrix: true class vs predicted probability ---
unique_classes = list(class_names)
mean_prob_matrix = np.zeros((len(unique_classes), len(unique_classes)))
for i, true_class in enumerate(unique_classes):
    mask = actual_events == true_class
    if mask.sum() > 0:
        mean_prob_matrix[i] = lr_probs[mask].mean(axis=0)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(mean_prob_matrix, cmap='RdYlGn', vmin=0, vmax=1)
for i in range(len(unique_classes)):
    for j in range(len(unique_classes)):
        ax.text(j, i, f'{mean_prob_matrix[i, j]:.2f}', ha='center', va='center',
                color='black' if mean_prob_matrix[i, j] > 0.3 else 'white', fontsize=11)
ax.set_xticks(range(len(unique_classes)))
ax.set_xticklabels(unique_classes, rotation=45, ha='right')
ax.set_yticks(range(len(unique_classes)))
ax.set_yticklabels(unique_classes)
ax.set_xlabel('Predicted Class (mean probability)')
ax.set_ylabel('True Class')
ax.set_title('Mean Prediction Probability by True Class')
fig.colorbar(im, ax=ax, label='Mean Probability')
plt.tight_layout()

# --- Per-trial summary table ---
rows = []
for i in range(len(actual_events)):
    row = {'Trial': i+1, 'True': actual_events[i], 'Predicted': prediction[i]}
    for j, cls in enumerate(class_names):
        row[f'P({cls})'] = f'{lr_probs[i, j]:.3f}'
    row['Correct'] = actual_events[i] == prediction[i]
    rows.append(row)
df = pd.DataFrame(rows)
print(f"Overall accuracy: {np.mean(actual_events == prediction):.2f}")
print(f"Mean max confidence: {lr_probs.max(axis=1).mean():.3f}")
df


Overall accuracy: 0.82
Mean max confidence: 0.987


,Trial,True,Predicted,P(ClosePalm),P(FixatedRest),P(LeftHand),P(RightHand),Correct
0,1,ClosePalm,ClosePalm,0.950,0.050,0.000,0.000,True
1,2,LeftHand,LeftHand,0.000,0.000,1.000,0.000,True
2,3,RightHand,RightHand,0.000,0.000,0.000,1.000,True
3,4,ClosePalm,ClosePalm,1.000,0.000,0.000,0.000,True
4,5,FixatedRest,FixatedRest,0.000,1.000,0.000,0.000,True
5,6,RightHand,RightHand,0.000,0.000,0.000,1.000,True
6,7,LeftHand,LeftHand,0.000,0.000,1.000,0.000,True
7,8,LeftHand,RightHand,0.000,0.000,0.000,1.000,False
8,9,ClosePalm,FixatedRest,0.005,0.995,0.000,0.000,False
9,10,ClosePalm,ClosePalm,1.000,0.000,0.000,0.000,True


In [21]:
# Pick the time window with best accuracy
best_idx = np.argmax(scores_windows[0])
cm, classes = confusion_matrices_per_window[0][best_idx]
plot_confusion_matrix(cm, classes, title=f"Confusion Matrix at t={w_times[best_idx]:.2f}s")

In [26]:
cm_avg, classes = plot_avg_confusion_matrix(
    confusion_matrices_per_window, w_times, 
    t_start=0, t_end=4.0
)

## Co-Adaptive Learning
After each new block, accumulate epochs and retrain the classifier with exponential block weights ג€” giving more influence to recent data.

**Workflow:**
1. After Block 1: run standard training (cells above) ג†’ `trained_clf`
2. After each subsequent block: load new epochs, run the cell below to retrain

In [112]:
# --- Initialise on first run (after Block 1) ---
# Stores epochs and trial counts per block for weighted retraining
if 'adaptive_block_epochs' not in dir():
    adaptive_block_epochs = [epochs.copy()]   # seed with Block 1 epochs
    print(f"Adaptive learning initialised with Block 1 ({len(epochs)} trials).")

Adaptive learning initialised with Block 1 (130 trials).


In [114]:
# --- Run this cell after each new block is recorded ---

# 1. Load the new block's XDF and preprocess
new_xdf_files = [f for f in recording_path.glob('*.xdf') if subject_name in f.name]
# TODO: update the slice below to select only the new block file(s)
new_xdf = new_xdf_files[-1:]   # <-- adjust as needed
print(f"Loading new block: {[f.name for f in new_xdf]}")

new_epochs_list = []
for xdf_file in new_xdf:
    raw = read_raw_xdf(xdf_file)
    _, epoch, _, _, _ = EEG_Preprocessing(current_path, raw, params_dict, pick_channels=True)
    epoch = remap_epoch_events_to_standard(epoch, standard_event_id, params_dict['desired_events'])
    params_dict['events_trigger_dict'] = {event: standard_event_id[event] for event in params_dict['desired_events']}
    new_epochs_list.append(epoch)

new_block_epochs = mne.concatenate_epochs(new_epochs_list, on_mismatch='warn')
adaptive_block_epochs.append(new_block_epochs)
print(f"Total blocks accumulated: {len(adaptive_block_epochs)}")

Loading new block: ['LD_MI3.xdf']
Creating RawArray with float64 data, n_channels=67, n_times=491013
    Range : 0 ... 491012 =      0.000 ...   982.024 secs
Ready.

###########################################################
removing subject specific bad electrodes from the raw data

###########################################################
removing bad channels from epochs:
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.

###########################################################
filtering the data
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 1e+02 Hz

IIR filter parameters
---------------------
Butterworth bandpass non-linear phase (one-pass forward) causal filter:
- Filter order 8 (forward)
- Cutoffs at 1.00, 100.00 Hz: -3.01, -3.01 dB

Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 49 - 51 Hz

IIR filter parameters
---------------------
Butterworth band

In [113]:
epochs

Number of events,130
Events,FixatedRest: 20LeftHand: 30MiddleHand: 30Rest: 20RightHand: 30
Time range,-5.000 – 6.000 s
Baseline,off


In [115]:
# --- Weighted retrain ---
alpha = 0.7   # decay factor: reduce to weight recent blocks more aggressively

# Combine all blocks and compute per-trial weights
all_epochs = mne.concatenate_epochs(adaptive_block_epochs, on_mismatch='warn')
block_counts = [len(e) for e in adaptive_block_epochs]
sample_weight = compute_block_weights(block_counts, alpha=alpha)

print(f"Block sizes: {block_counts}")
print(f"Block weights (normalised): {[round(sample_weight[sum(block_counts[:i])], 3) for i in range(len(block_counts))]}")

# Crop to classifier window and augment
epochs_copy = all_epochs.copy()
returned_dict_adaptive = crop_the_data(
    epochs_copy, np.arange(len(all_epochs)), [],
    params_dict['classifier_window_s'], params_dict['classifier_window_e'],
    use_all_for_training=True
)
train_x = returned_dict_adaptive['train_set_data']
train_y = returned_dict_adaptive['train_set_labels']

# Augment
aug_x, aug_y = augment_data(params_dict['augmentation_params'], train_x, train_y, all_epochs.info['sfreq'])

# Expand weights to match augmented dataset, then resample
aug_factor = len(aug_x) // len(train_x)
remainder  = len(aug_x) %  len(train_x)
aug_weights = np.concatenate([np.tile(sample_weight, aug_factor),
                               sample_weight[:remainder]])

# Resample data proportional to weights (works with any estimator)
aug_x_resampled, aug_y_resampled = resample_by_weights(aug_x, aug_y, aug_weights)
print(f"Resampled dataset: {len(aug_y_resampled)} trials (from {len(aug_y)} augmented)")

# Retrain
trained_clf, csp, lda = classifier_training(aug_x_resampled, aug_y_resampled, params_dict)
print(f"Classifier retrained on {len(all_epochs)} trials across {len(adaptive_block_epochs)} blocks.")

Not setting metadata
239 matching events found
No baseline correction applied
Block sizes: [130, 109]
Block weights (normalised): [0.836, 1.195]
Resampled dataset: 239 trials (from 239 augmented)
Classifier retrained on 239 trials across 2 blocks.


In [116]:
# --- Quick sanity check on the updated classifier ---
scores_windows, confusion_matrices_per_window, w_times = sanity_check_trained_clf(
    trained_clf, all_epochs, params_dict
)
plot_accuracy_over_time(scores_windows, w_times, params_dict)
plot_average_confusion_fixed_cv(confusion_matrices_per_window, w_times, t_start=2, t_end=5)

(array([[0.86410256, 0.02307692, 0.09487179, 0.00769231, 0.01025641],
        [0.02393162, 0.88205128, 0.05811966, 0.01538462, 0.02051282],
        [0.00854701, 0.03589744, 0.93162393, 0.0034188 , 0.02051282],
        [0.00935551, 0.01039501, 0.0956341 , 0.86070686, 0.02390852],
        [0.01025641, 0.03247863, 0.06153846, 0.02735043, 0.86837607]]),
 array(['FixatedRest', 'LeftHand', 'MiddleHand', 'Rest', 'RightHand'],
       dtype='<U11'))

In [117]:
plot_average_confusion_fixed_cv(confusion_matrices_per_window, w_times, t_start=-2, t_end=0)

(array([[0.12962963, 0.00740741, 0.19259259, 0.54814815, 0.12222222],
        [0.0691358 , 0.10123457, 0.20740741, 0.52098765, 0.10123457],
        [0.07654321, 0.07160494, 0.30617284, 0.47407407, 0.07160494],
        [0.15165165, 0.26426426, 0.29279279, 0.07207207, 0.21921922],
        [0.02222222, 0.05432099, 0.30864198, 0.48888889, 0.12592593]]),
 array(['FixatedRest', 'LeftHand', 'MiddleHand', 'Rest', 'RightHand'],
       dtype='<U11'))

Save the trained model 


In [ ]:
fname = 'ID_Raw_for_analysis_240526'
path_fname = current_path /'Models'/ fname

#create a pickle file
picklefile = open(path_fname, 'wb')
#pickle the dictionary and write it to file
pickle.dump(Raw_for_analysis, picklefile)
#close the file
picklefile.close()

# === Saving ===
ann_fname = current_path / 'Models' / 'ID_ann_240526'
with open(ann_fname, 'wb') as f:
    pickle.dump(event_ann, f)
# === Saving ===
with open(ann_fname, 'wb') as f:
    pickle.dump(event_ann, f)



In [73]:
picks=params_dict['Electorde_Group']


import joblib
# === Saving ===
channels_fname = current_path / 'Models' / 'channels_list_LD_030626.pkl'
model_fname = current_path / 'Models' / 'ts_model_LD_030626.joblib'
with open(channels_fname, 'wb') as f:
    pickle.dump(picks, f)

joblib.dump(clf, model_fname)


# %%

Saved_Model = trained_clf
fname = 'LD'+'ts_model_030626'
path_fname = current_path /'Models'/ fname

#create a pickle file
picklefile = open(path_fname, 'wb')
#pickle the dictionary and write it to file
pickle.dump(Saved_Model, picklefile)
#close the file
picklefile.close()



picks=params_dict['Electorde_Group']
fname = 'electrode_picks-LD_030626'
path_fname = current_path /'Models'/ fname

#create a pickle file
picklefile = open(path_fname, 'wb')
#pickle the dictionary and write it to file
pickle.dump(picks, picklefile)
#close the file
picklefile.close()


fname = 'mean-LD_030626'
path_fname = current_path /'Models'/ fname

#create a pickle file
picklefile = open(path_fname, 'wb')
#pickle the dictionary and write it to file
pickle.dump(mean_across_epochs, picklefile)
#close the file
picklefile.close()

params_dict
fname = 'params_dict_LD_030626'
path_fname = current_path /'Models'/ fname

#create a pickle file
picklefile = open(path_fname, 'wb')
#pickle the dictionary and write it to file
pickle.dump(params_dict, picklefile)
#close the file
picklefile.close()

In [85]:
picks=params_dict['Electorde_Group']


import joblib
# === Saving ===
channels_fname = current_path / 'Models' / 'channels_list_LD_030626.pkl'
model_fname = current_path / 'Models' / 'Bi_ts_model_LD_030626.joblib'
with open(channels_fname, 'wb') as f:
    pickle.dump(picks, f)

joblib.dump(clf, model_fname)


# %%

Saved_Model = trained_clf
fname = 'LD'+'Bi_ts_model_030626'
path_fname = current_path /'Models'/ fname

#create a pickle file
picklefile = open(path_fname, 'wb')
#pickle the dictionary and write it to file
pickle.dump(Saved_Model, picklefile)
#close the file
picklefile.close()



picks=params_dict['Electorde_Group']
fname = 'electrode_picks-LD_030626'
path_fname = current_path /'Models'/ fname

#create a pickle file
picklefile = open(path_fname, 'wb')
#pickle the dictionary and write it to file
pickle.dump(picks, picklefile)
#close the file
picklefile.close()


fname = 'mean-Bi_LD_030626'
path_fname = current_path /'Models'/ fname

#create a pickle file
picklefile = open(path_fname, 'wb')
#pickle the dictionary and write it to file
pickle.dump(mean_across_epochs, picklefile)
#close the file
picklefile.close()

params_dict
fname = 'params_dict_Bi_LD_030626'
path_fname = current_path /'Models'/ fname

#create a pickle file
picklefile = open(path_fname, 'wb')
#pickle the dictionary and write it to file
pickle.dump(params_dict, picklefile)
#close the file
picklefile.close()

In [ ]:
picks=params_dict['Electorde_Group']


import joblib
# === Saving ===
channels_fname = current_path / 'Models' / 'channels_list_ID_240526.pkl'
model_fname = current_path / 'Models' / 'ts_model_ID_240526.joblib'
with open(channels_fname, 'wb') as f:
    pickle.dump(picks, f)

joblib.dump(clf, model_fname)


# %%

Saved_Model = trained_clf
fname = 'ID'+'csp_model_240526'
path_fname = current_path /'Models'/ fname

#create a pickle file
picklefile = open(path_fname, 'wb')
#pickle the dictionary and write it to file
pickle.dump(Saved_Model, picklefile)
#close the file
picklefile.close()



picks=params_dict['Electorde_Group']
fname = 'electrode_picks-csp240526'
path_fname = current_path /'Models'/ fname

#create a pickle file
picklefile = open(path_fname, 'wb')
#pickle the dictionary and write it to file
pickle.dump(picks, picklefile)
#close the file
picklefile.close()


fname = 'mean-ID_240526'
path_fname = current_path /'Models'/ fname

#create a pickle file
picklefile = open(path_fname, 'wb')
#pickle the dictionary and write it to file
pickle.dump(mean_across_epochs, picklefile)
#close the file
picklefile.close()

params_dict
fname = 'params_dict-Restcsp_Id_240526'
path_fname = current_path /'Models'/ fname

#create a pickle file
picklefile = open(path_fname, 'wb')
#pickle the dictionary and write it to file
pickle.dump(params_dict, picklefile)
#close the file
picklefile.close()

In [ ]:

scaler = RobustScaler()
all_epoch_data_flat = all_epoch_data.reshape(all_epoch_data.shape[0], -1)
all_epoch_data_normalized = scaler.fit_transform(all_epoch_data_flat).reshape(all_epoch_data.shape)

# Update combined epochs with normalized data
epochs._data = all_epoch_data_normalized

In [ ]:
from sklearn.preprocessing import StandardScaler, RobustScaler
